# LoRA SFT v9 — v8 stack × 3 epochs (test undertraining hypothesis)

**What v9 tests:** Tong did ~1000 steps at batch 32 = ~32k example-views (≈1 epoch over his ~32k corpus). v8 did 146 steps at batch 32 = ~4.7k example-views (1 epoch over our 4681 records). He saw **~7× more gradient signal**.

We can't easily build a 32k corpus right now, but we CAN run more epochs. v9 runs the same v8 stack for **3 epochs (≈438 steps)** — a 3× increase in training intensity at our current data scale.

**Hypothesis:** if v8's 0.56 was due to undertraining, v9 should lift toward or past v3's 0.58 baseline. If v9 stays ≤ 0.58, the bottleneck is genuinely data quality, not training intensity, and the SFT branch is exhausted.

**Why 3 epochs not 5/7:**
- 3 epochs ≈ 14k example-views — closer to Tong without going crazy
- 5-7 epochs risks overfitting since v8's 1-epoch training loss already ended near 0.05
- 3 epochs at ~22 sec/step × 438 steps ≈ 160 min ≈ ~2.7 hr — comfortable budget

**Single variable changed vs v8:** `NUM_EPOCHS = 3` (was 1). Everything else unchanged — same Unsloth + CCE stack, same data, same hyperparams.

**Reference (6 SFT submissions, all 0.56-0.58):**
- v3 baseline: **0.58** ← high-water mark
- v4 gravity fix: 0.56
- v5 synth-tr: 0.56
- v6 broken MoE tying: 0.58
- v7 correct MoE tying: 0.58
- v8 Unsloth + CCE: 0.56
- Tong: 0.85

**Expected outcomes:**
- v9 ≥ 0.60: undertraining was the issue; multi-epoch is our path
- v9 ~ 0.56-0.58: data quality is the real ceiling; SFT branch genuinely exhausted
- v9 < 0.56: 3 epochs overfit at this data scale

In [ ]:
# Cell 0: Config + smoke-test toggle
SMOKE_TEST = False      # True = 10 steps on 100 examples for env validation; False = full run

# Architecture (identical to v8)
LORA_RANK = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0.0
TARGET_MODULES = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'in_proj', 'out_proj',
    'up_proj', 'down_proj',
    'lm_head',
]
MOE_TIE_WEIGHTS = True

# Training
MAX_SEQ_LEN = 2048
BATCH_SIZE = 32
MICRO_BATCH_SIZE = 4
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3         # *** v9 ONLY CHANGE *** (v8 used 1) — tests undertraining hypothesis
SEED = 42

# Smoke test overrides
SMOKE_N_EXAMPLES = 100
SMOKE_N_STEPS = 10
if SMOKE_TEST:
    print('*** SMOKE TEST MODE *** — will train 10 steps on 100 examples only')

In [ ]:
# Cell 1: Install from offline-deps datasets (matches Tong's working setup)
import os, subprocess, sys, time

T0 = time.perf_counter()
def step(label):
    print(f'[{time.perf_counter() - T0:7.1f}s] {label}', flush=True)

# Main packages from mayukh18's offline package collection (Tong's source)
subprocess.run(
    'pip install -q --no-index --find-links /kaggle/input/datasets/mayukh18/nemotron-packages/packages '
    'unsloth trl peft transformers datasets accelerate bitsandbytes',
    shell=True, check=True,
)
# Pre-built wheels for the Mamba CUDA fast-path
subprocess.run(
    'pip install -q /kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl',
    shell=True, check=True,
)
subprocess.run(
    'pip install -q /kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl',
    shell=True, check=True,
)
# RTX-specific wheels (best-effort)
for _wd in ['/kaggle/input/datasets/llkh0a/rtx-wheels/wheels']:
    if os.path.isdir(_wd):
        subprocess.run(
            ['pip', 'install', '-q', '--no-index', '--find-links', _wd,
             'protobuf==6.33.5', 'sentencepiece', 'safetensors', 'huggingface_hub'],
            check=False,
        )
subprocess.run('rm -rf /kaggle/tmp/*', shell=True, check=False)
step('Packages installed')

import torch, random, numpy as np
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
step(f'Seeds set to {SEED}')

# Sanity check on critical imports BEFORE model load (fail fast if env is broken)
import unsloth, peft, transformers, accelerate
from unsloth import FastLanguageModel
import causal_conv1d, mamba_ssm
from cut_cross_entropy import linear_cross_entropy
print(f'unsloth={unsloth.__version__}, peft={peft.__version__}, transformers={transformers.__version__}')
print(f'causal_conv1d={causal_conv1d.__version__}, mamba_ssm={mamba_ssm.__version__}')
print(f'cut_cross_entropy: imported OK')
step('All critical imports OK — env is healthy')

In [ ]:
# Cell 2: Load Nemotron 3 Nano via Unsloth's FastLanguageModel + wrap in LoRA
import kagglehub
from peft import LoraConfig
from peft.tuners.lora import Linear as LoraLinear

MODEL_PATH = kagglehub.model_download('metric/nemotron-3-nano-30b-a3b-bf16/transformers/default')
step(f'Model path: {MODEL_PATH}')

t = time.perf_counter()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=False,
    load_in_8bit=False,
    full_finetuning=False,
    trust_remote_code=True,
    unsloth_force_compile=True,
    attn_implementation='eager',
    dtype=torch.bfloat16,
)
step(f'Model loaded in {time.perf_counter() - t:.0f}s, VRAM={torch.cuda.memory_allocated() / 1e9:.1f} GB')

IM_END_ID = tokenizer.convert_tokens_to_ids('<|im_end|>')
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
step(f'Tokenizer: IM_END_ID={IM_END_ID}, eos={tokenizer.eos_token_id}, pad={tokenizer.pad_token_id}')

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)
FastLanguageModel.for_training(model)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
step(f'LoRA: {trainable:,} trainable / {total:,} total ({100*trainable/total:.2f}%)')

# Patch Mamba CUDA fast-path (Tong's trick — the default is False, but we have causal_conv1d+mamba_ssm)
nemotron_mod = None
for _name, _m in sys.modules.items():
    if 'modeling_nemotron_h' in _name and hasattr(_m, 'is_fast_path_available'):
        nemotron_mod = _m
        break
if nemotron_mod is None:
    raise RuntimeError('Could not find modeling_nemotron_h module to patch')
print(f'is_fast_path_available was: {nemotron_mod.is_fast_path_available}')
nemotron_mod.is_fast_path_available = True
print('Patched is_fast_path_available = True')

# Manually add lm_head LoRA (Unsloth drops it for MoE models)
_causal_lm = model
while hasattr(_causal_lm, 'model'):
    _causal_lm = _causal_lm.model
_lm_head = _causal_lm.lm_head
if not isinstance(_lm_head, LoraLinear):
    _cfg = LoraConfig(r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT)
    model.base_model._create_and_replace(
        _cfg, 'default', target=_lm_head, target_name='lm_head', parent=_causal_lm,
    )
    print('Manually added LoRA to lm_head')
else:
    print('lm_head already has LoRA')

# Cast LoRA params to fp32 (base stays bf16; MoE router stays fp32 — Nemotron loads it that way)
for name, param in model.named_parameters():
    if '.lora_' in name:
        param.data = param.data.to(torch.float32)
for name, param in model.named_parameters():
    if '.lora_' in name:
        assert param.dtype == torch.float32, f'{name} expected fp32 got {param.dtype}'
    elif '.mixer.gate.' in name:
        assert param.dtype == torch.float32, f'{name} expected fp32 (router) got {param.dtype}'
    else:
        assert param.dtype == torch.bfloat16, f'{name} expected bf16 got {param.dtype}'
step('LoRA cast to fp32, dtype invariants verified')

In [ ]:
# Cell 3: Patch CausalLM forward with Cut Cross-Entropy (no logits materialization)
_base = model
while hasattr(_base, 'model'):
    _base = _base.model

def _patched_causal_forward(input_ids=None, attention_mask=None, labels=None, **kwargs):
    backbone_out = _base.backbone(
        input_ids=input_ids,
        attention_mask=attention_mask,
        **{k: v for k, v in kwargs.items() if k in ('position_ids', 'past_key_values', 'use_cache')},
    )
    hidden_states = backbone_out[0]
    lm_head = _base.lm_head
    base_w = lm_head.base_layer.weight
    lora_A = lm_head.lora_A['default'].weight
    lora_B = lm_head.lora_B['default'].weight
    scaling = lm_head.scaling['default']
    lm_weight = base_w + scaling * lora_B @ lora_A
    if labels is not None:
        per_token_ce = linear_cross_entropy(hidden_states, lm_weight, labels, reduction='none')
        loss = per_token_ce.mean()
    else:
        per_token_ce = None
        loss = None
    model._cached_per_token_ce = per_token_ce
    return loss

_base.forward = _patched_causal_forward
step('Patched CausalLM.forward with Cut Cross-Entropy')

In [ ]:
# Cell 4: MoE expert weight tying (Unsloth uses batched expert tensors so Tong's original code works)
moe_tied_params = []
if MOE_TIE_WEIGHTS:
    w1_proj_names = ('gate_up_proj', 'up_proj', 'gate_proj', '.w1.')
    w2_proj_names = ('down_proj', '.w2.')
    for name, param in model.named_parameters():
        if not param.requires_grad: continue
        if '.experts.' not in name or '.lora_' not in name: continue
        is_w1 = any(p in name for p in w1_proj_names)
        is_w2 = any(p in name for p in w2_proj_names)
        is_A = '.lora_A.' in name
        is_B = '.lora_B.' in name
        should_tie = (is_w1 and is_A) or (is_w2 and is_B)
        if not should_tie: continue
        if param.dim() < 2 or param.shape[0] <= 1: continue
        moe_tied_params.append((name, param))
    
    print(f'MoE tying: {len(moe_tied_params)} batched-expert LoRA params identified')
    for name, p in moe_tied_params[:3]:
        print(f'  {name}  shape={tuple(p.shape)}')
    if len(moe_tied_params) > 3:
        print(f'  ... and {len(moe_tied_params) - 3} more')
    
    if not moe_tied_params:
        print('⚠️ No batched-expert LoRA found — Unsloth may have wrapped differently than expected.')
        print('   Proceeding without MoE tying.')
    else:
        # Init tied (mean across expert dim)
        with torch.no_grad():
            for _name, p in moe_tied_params:
                mean = p.data.mean(dim=0, keepdim=True)
                p.data.copy_(mean.expand_as(p.data))
        # Verify
        for _name, p in moe_tied_params:
            ref = p.data[0]
            for i in range(1, p.shape[0]):
                assert torch.equal(p.data[i], ref), f'tying init failed at {_name}, slice {i}'
        step(f'MoE expert LoRAs initialized + verified tied ({len(moe_tied_params)} param groups)')

def tie_moe_grads():
    '''Sum gradients across expert dim before optimizer step.'''
    if not moe_tied_params: return
    with torch.no_grad():
        for _name, p in moe_tied_params:
            if p.grad is None: continue
            grad_sum = p.grad.sum(dim=0, keepdim=True)
            p.grad.copy_(grad_sum.expand_as(p.grad))
step('MoE grad-tying function ready')

In [ ]:
# Cell 5: Load OUR training data + tokenize (v3 corpus, byte-identical)
import json, glob

PROMPT_SUFFIX = (
    '\nPlease put your final answer inside `\\boxed{}`. '
    'For example: `\\boxed{your answer}`'
)

CANDIDATE_PATHS = [
    '/kaggle/input/datasets/gideonnazarian/nemotron-solver-training-data/training-data.jsonl',
    '/kaggle/input/nemotron-solver-training-data/training-data.jsonl',
]
DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p): DATA_PATH = p; break
if DATA_PATH is None:
    hits = glob.glob('/kaggle/input/**/training-data.jsonl', recursive=True)
    if hits: DATA_PATH = hits[0]
if DATA_PATH is None:
    raise FileNotFoundError(f'training-data.jsonl not found')
step(f'Loading data from {DATA_PATH}')

records = []
with open(DATA_PATH) as f:
    for line in f:
        records.append(json.loads(line))
step(f'Loaded {len(records)} training records')

if SMOKE_TEST:
    records = records[:SMOKE_N_EXAMPLES]
    step(f'SMOKE_TEST: using first {len(records)} records only')

def tokenize_example(record):
    user_text = record['prompt'] + PROMPT_SUFFIX
    prompt_text = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': user_text}],
        tokenize=False, add_generation_prompt=True,
    )
    prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
    response_text = (
        record['reasoning'] + '\n</think>\n\\boxed{' + record['answer'] + '}<|im_end|>'
    )
    response_ids = tokenizer.encode(response_text, add_special_tokens=False)
    full_ids = prompt_ids + response_ids
    if len(full_ids) > MAX_SEQ_LEN:
        full_ids = full_ids[:MAX_SEQ_LEN]
    # Labels: -100 for prompt portion (masked), real ids for response portion
    n_prompt = len(prompt_ids)
    labels = [-100] * n_prompt + full_ids[n_prompt:]
    labels = labels[:len(full_ids)]
    return {
        'input_ids': full_ids[:-1],     # CCE expects targets shifted, so do tokens[:-1] / labels[1:]
        'targets':   full_ids[1:],
        'weights':   [0.0 if l == -100 else 1.0 for l in labels[1:]],
    }

examples = []
for rec in records:
    examples.append(tokenize_example(rec))
step(f'Tokenized {len(examples)} examples')

lens = [len(e['input_ids']) for e in examples]
step(f'Token-length stats: min={min(lens)}, median={sorted(lens)[len(lens)//2]}, max={max(lens)}')

In [ ]:
# Cell 6: Custom training loop with multi-epoch support
import gc, math
gc.collect(); torch.cuda.empty_cache()

# Build a multi-epoch index sequence. For v9 with NUM_EPOCHS=3, this gives
# us len(examples) * 3 records to iterate over before the outer loop ends.
# We keep the same order each epoch (matches Tong's SHUFFLE_DATASET=False).
indices_one_epoch = list(range(len(examples)))
indices = indices_one_epoch * NUM_EPOCHS
steps_per_epoch = max(1, len(examples) // BATCH_SIZE)
num_steps = SMOKE_N_STEPS if SMOKE_TEST else steps_per_epoch * NUM_EPOCHS

step(f'Training: {num_steps} steps over {NUM_EPOCHS} epoch(s), batch={BATCH_SIZE}, micro_batch={MICRO_BATCH_SIZE}, lr={LEARNING_RATE}')
step(f'  ({steps_per_epoch} steps/epoch * {NUM_EPOCHS} epochs; example-views per step = {BATCH_SIZE})')

optimizer = None
device = next(model.parameters()).device
training_log = []

def _log(msg):
    print(msg, flush=True)
    training_log.append(msg)

step_n = 0
for batch_start in range(0, len(indices), BATCH_SIZE):
    if step_n >= num_steps: break
    batch_idx = indices[batch_start:batch_start + BATCH_SIZE]
    batch = [examples[i] for i in batch_idx]
    n = len(batch)
    n_accum = math.ceil(n / MICRO_BATCH_SIZE)
    total_loss_sum = 0.0
    total_weight_sum = 0.0
    t_step = time.time()
    
    for mb_start in range(0, n, MICRO_BATCH_SIZE):
        mb = batch[mb_start:mb_start + MICRO_BATCH_SIZE]
        max_len = max(len(e['input_ids']) for e in mb)
        n_micro = len(mb)
        
        pad_id = tokenizer.pad_token_id
        input_ids = torch.zeros(n_micro, max_len, dtype=torch.long, device=device)
        targets   = torch.zeros(n_micro, max_len, dtype=torch.long, device=device)
        weights   = torch.zeros(n_micro, max_len, dtype=torch.float32, device=device)
        attn_mask = torch.zeros(n_micro, max_len, dtype=torch.long, device=device)
        for i, e in enumerate(mb):
            L = len(e['input_ids'])
            input_ids[i, :L] = torch.tensor(e['input_ids'], dtype=torch.long)
            targets  [i, :L] = torch.tensor(e['targets'],   dtype=torch.long)
            weights  [i, :L] = torch.tensor(e['weights'],   dtype=torch.float32)
            attn_mask[i, :L] = 1
        
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            model(input_ids=input_ids, attention_mask=attn_mask, labels=targets, use_cache=False)
            per_token_ce = model._cached_per_token_ce
            weighted = per_token_ce * weights
            w_sum = weights.sum()
            l_sum = weighted.sum()
            loss = l_sum / w_sum if w_sum > 0 else l_sum * 0.0
        
        (loss / n_accum).backward()
        total_loss_sum += l_sum.item()
        total_weight_sum += w_sum.item()
        del loss, per_token_ce, weighted
    
    if optimizer is None:
        optimizer = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=LEARNING_RATE, betas=(0.9, 0.95), eps=1e-8, weight_decay=0.0,
        )
    # Linear LR decay over ALL steps (across epochs)
    lr = LEARNING_RATE * (1 - step_n / num_steps)
    for pg in optimizer.param_groups:
        pg['lr'] = lr
    
    tie_moe_grads()  # Average MoE expert grads BEFORE optimizer step
    grad_norm = torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], max_norm=1e9,
    )
    optimizer.step()
    optimizer.zero_grad()
    
    loss_mean = total_loss_sum / total_weight_sum if total_weight_sum > 0 else 0
    step_n += 1
    peak_gb = torch.cuda.max_memory_allocated() / 1e9
    epoch_n = (step_n - 1) // steps_per_epoch + 1
    _log(f'  step {step_n}/{num_steps} (epoch {epoch_n}/{NUM_EPOCHS}): loss={loss_mean:.4f} grad_norm={grad_norm:.3f} lr={lr:.2e} wall={time.time()-t_step:.1f}s peak={peak_gb:.1f}GB')

step(f'Training done ({step_n} steps). Peak VRAM: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB')

In [ ]:
# Cell 7: Save adapter + rename lm_head keys (Tong's grader-compat trick)
from safetensors.torch import load_file, save_file
import zipfile

OUTPUT_DIR = '/kaggle/working'
for f in os.listdir(OUTPUT_DIR):
    if f.startswith('adapter') or f.endswith('.safetensors') or f.endswith('.bin'):
        try: os.remove(os.path.join(OUTPUT_DIR, f))
        except (IsADirectoryError, PermissionError): pass

model.save_pretrained(OUTPUT_DIR)
step('Adapter saved')

# Rename lm_head key (Tong: adapter must reference 'backbone.lm_head' not 'lm_head' for the grader)
st_path = os.path.join(OUTPUT_DIR, 'adapter_model.safetensors')
if os.path.isfile(st_path):
    tensors = load_file(st_path)
    renamed = {
        k.replace('base_model.model.lm_head.', 'base_model.model.backbone.lm_head.'): v
        for k, v in tensors.items()
    }
    save_file(renamed, st_path)
    step('lm_head keys renamed for grader compatibility')

# Clean unsloth compiled cache so it doesn't bloat the workspace
_ucache = 'unsloth_compiled_cache'
if os.path.isdir(_ucache):
    import shutil; shutil.rmtree(_ucache)

if SMOKE_TEST:
    step('SMOKE_TEST=True — SKIPPING submission.zip creation (adapter is for inspection only)')
    print('Smoke test complete. If loss is decreasing and no errors, re-push with SMOKE_TEST=False.')
else:
    adapter_files = [f for f in os.listdir(OUTPUT_DIR) if f.startswith('adapter')]
    SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, 'submission.zip')
    with zipfile.ZipFile(SUBMISSION_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
        for fname in adapter_files:
            zf.write(os.path.join(OUTPUT_DIR, fname), fname)
    for fname in adapter_files:
        os.remove(os.path.join(OUTPUT_DIR, fname))
    step(f'Wrote {SUBMISSION_ZIP}')

step(f'Total wall time: {(time.perf_counter() - T0) / 60:.1f} min')
print('Done.')